Question 1: Design the neural network

In [ ]:
import torch
import torch.nn as nn

# Step 1: Layer Dimensions
input_dim = 5
hidden_dim = 10
output_dim = 3

# Step 2: Activation Function
def activation_function(x):
    return torch.relu(x)  # Using ReLU activation function

# Step 3: Neural Network Architecture
class ThreeLayerNN(nn.Module):
    def __init__(self):
        super(ThreeLayerNN, self).__init__()
        # 5 -> 10
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        # 10 -> 10
        self.hidden_layer = nn.Linear(hidden_dim, hidden_dim)
        # 10 -> 3
        self.output_layer = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # a0 = x dimension 5
        # z1 = W1a0 + b1 W1 is 10 by 5 and b1 is 10 by 1 ... So z1 is 10 dimensional
        # a1= relu(z1) = max(z1, 0)
        # relu returns max(0,x)
        # z2 = W2*a1+b2
        # a2 = relu(z2)
        # z3 = W3*a2 + b3 where W3 is 3 by 10 and b3 is 3 by 1
        # z3 is 3 by 1
        # z3 = (a, b, c) ... logits
        # p0 = (e^a / (e^a + e^b + e^c),
        #   p1 = e^b / (e^a + e^b + e^c),
        #   p2 = e^c / (e^a + e^b + e^c))

        # (a, b) -> (softmax(a, b))

        # a -> sigmoid(a) .... (p0 = 1 - sigmoid(a), p1 = sigmoid(a))


        # 10 by 1
        hidden_output = activation_function(self.input_layer(x))

        # 10 by 1
        hidden_output = activation_function(self.hidden_layer(hidden_output))

        # 3 by 1
        output = self.output_layer(hidden_output)
        # softmax
        return torch.softmax(output,dim=1)

# Step 4: Initialization and Testing
x = torch.randn(1, input_dim)
model = ThreeLayerNN()
output = model(x)
print("Output tensor:", output)

# (x, (0, 1))
# (0, 1) = (y0, y1)
# L = -y0 log(p0) -y1 log(p1)
# L = - log(p1) ... We want to minimize this



Output tensor: tensor([[0.4971, 0.3069, 0.1960]], grad_fn=<SoftmaxBackward0>)


Question 2: Autograd

In [ ]:
import torch

# Define the real function f(x) = (x^4 - 3x^3 + 2x^2 - 5x + 7) / (2x^3 + 5x^2 - x + 3)
def real_function(x):
    result = (x**4 - 3*x**3 + 2*x**2 - 5*x + 7) / (2*x**3 + 5*x**2 - x + 3)
    return result

# Define a real number for which to compute the gradient
x = torch.tensor(1.0, requires_grad=True, dtype=torch.float32)

# Compute the real function f(x)
result = real_function(x)

# x -> result = real_function(x)

# Compute the gradient of f(x) with respect to x
# df / fx
result.backward()

# Obtain the gradient of x
gradient_x = x.grad

print("Gradient of x:", gradient_x)



"""

a, b, c (parameters)
x is data

z1 = ax+b
a1 = sigmoid(z1)

z2 = c(a1) + a

dz2/da
dz2/db
dz2/dc

a        b    c------------------------------------
 -       -                                        -
  - --   -                                         -
      -  -                                          -
x ---> (z1 = ax+b) --> (a1 = sigmoid(z1)) --> (z2 = c(a1) + a)


dz2/da = dz2/da + dz2/da1 * da1/dz1 * dz1/da




"""


Gradient of x: tensor(-1.0370)


'\n\na, b, c (parameters)\nx is data\n\nz1 = ax+b\na1 = sigmoid(z1)\n\nz2 = c(a1) + a\n\ndz2/da\ndz2/db\ndz2/dc\n\na        b    c------------------------------------\n -       -                                        -\n  - --   -                                         -\n      -  -                                          -\nx ---> (z1 = ax+b) --> (a1 = sigmoid(z1)) --> (z2 = c(a1) + a)\n\n\ndz2/da = dz2/da + dz2/da1 * da1/dz1 * dz1/da\n\n\n\n\n'

Optional Question: Logistic Regression

In [ ]:
# Assignment Question 1:
import torch
import torch.nn as nn
import torch.optim as optim

# Step 1: Prepare the data
# Define the number of features (input size)
input_size = 2

# Define the number of classes (output size)
output_size = 1

# Create the training dataset
X_train = torch.tensor([[1.0, 2.0], [2.0, 3.0], [3.0, 4.0], [5.0, 1.0]])
y_train = torch.tensor([0, 0, 1, 1])  # Binary classification labels (0 or 1)

# Step 2: Define the logistic regression model
class LogisticRegressionModel(nn.Module):
    def __init__(self):
        super(LogisticRegressionModel, self).__init__()
        # W1, b1
        # (2, 2), (2, 1)
        self.linear = nn.Linear(input_size, output_size)

    def forward(self, x):
        # Notice: no probabilities here!
        return torch.sigmoid(self.linear(x))

model = LogisticRegressionModel()

# Step 3: Define the loss function and optimizer
criterion = nn.BCELoss()  # CrossEntropyLoss is used for binary classification

#

# theta_new = theta_old - lr * dL/dtheta(theta_old)
optimizer = optim.SGD(model.parameters(), lr=2.0)

# Step 4: Train the model
num_epochs = 1000
for epoch in range(num_epochs):
    # Forward pass
    outputs = model(X_train)
    loss = criterion(outputs.squeeze(), y_train.float())

    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

# Step 5: Test the model
X_test = torch.tensor([[4.0, 3.0]])
predicted = model(X_test)

# (v1, v2)
# (-10, 5)
# (e^{v1}/(e^v1+e^{v2}), e^{v2} / (e^v1+e^v2))
predicted_class = torch.argmax(predicted).item()
print("Predicted class:", predicted_class)


Epoch [100/1000], Loss: 0.0411
Epoch [200/1000], Loss: 0.0316
Epoch [300/1000], Loss: 0.0257
Epoch [400/1000], Loss: 0.0216
Epoch [500/1000], Loss: 0.0186
Epoch [600/1000], Loss: 0.0163
Epoch [700/1000], Loss: 0.0146
Epoch [800/1000], Loss: 0.0131
Epoch [900/1000], Loss: 0.0120
Epoch [1000/1000], Loss: 0.0110
Predicted class: 0


Optional Question: Attention with Einsum

In [ ]:
import torch
import torch.nn.functional as F

def single_head_attention_with_einsum(query, key, value):
    # Calculate the dot product between query and key using einsum
    attention_scores = torch.einsum('bqd,bkd->bqk', query, key)

    # Scale the dot product by dividing it by the square root of the dimension of the key vector
    scaled_attention_scores = attention_scores / (key.size(-1) ** 0.5)

    # Apply softmax to obtain attention weights along the last dimension (key dimension)
    attention_weights = F.softmax(scaled_attention_scores, dim=-1)

    # Compute the weighted sum of value vectors using einsum
    attended_values = torch.einsum('bqk,bkd->bqd', attention_weights, value)

    return attended_values, attention_weights

query = torch.randn(3,2,4)
key = torch.randn(3,5,4)
value = torch.randn(3,5,4)

values, weights = single_head_attention_with_einsum(query, key, value)
print(values.shape,weights.shape)
print(values, weights)


torch.Size([3, 2, 4]) torch.Size([3, 2, 5])
tensor([[[-0.2515, -0.3775, -0.6201, -0.6960],
         [ 0.3799,  2.3774,  0.8009, -0.2097]],

        [[ 0.5267, -0.2937, -0.4736,  0.4247],
         [ 1.0051, -0.8045, -0.1286,  0.3433]],

        [[ 0.4079, -0.2926, -0.7999,  0.4843],
         [ 0.0866, -0.2562, -0.6483,  0.6478]]]) tensor([[[0.2977, 0.0035, 0.0370, 0.0340, 0.6278],
         [0.0184, 0.8890, 0.0586, 0.0118, 0.0221]],

        [[0.4556, 0.0705, 0.1207, 0.1221, 0.2311],
         [0.0707, 0.2291, 0.4999, 0.0712, 0.1292]],

        [[0.2419, 0.0773, 0.5201, 0.0689, 0.0918],
         [0.0531, 0.2772, 0.3421, 0.1835, 0.1441]]])
